# Exploración Inicial — Señales EEG/EMG

Este notebook analiza el dataset `datos_eeg_emg.csv` generado desde `Data.txt`.

**Estructura del dataset:**
| Columna | Tipo | Descripción |
|---------|------|-------------|
| `Tiempo_s` | float | Vector de tiempo en segundos (0.001 s de paso → 1000 Hz) |
| `EEG_1` | float | Canal EEG principal (~1–2 µV) |
| `EEG_2` | float | Canal EEG secundario (~1–2 µV) |
| `EMG_1..6` | float | Canales EMG musculares (~0.02 µV, escala menor) |

**Canales clave según `app_edu.py`:**
- Para detección de **marcadores de movimiento**: un canal **EMG** (burst detector)
- Para análisis de **BP y desincronización beta**: los canales **EEG**

> **Nota reunión 09-04:** el pipeline es → detectar bursts EMG → usar como marcadores → cortar EEG → buscar BP (onda lenta pre-movimiento) y caída en banda Beta (13–30 Hz)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from scipy.stats import kurtosis, skew
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (14, 4)

DATA_PATH = '../data/datos_eeg_emg.csv'
SRATE = 1000  # Hz

## 1. Carga y descripción general

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'Muestras  : {len(df):,}')
print(f'Columnas  : {list(df.columns)}')
print(f'Duración  : {df.Tiempo_s.iloc[-1]:.1f} s  ({df.Tiempo_s.iloc[-1]/60:.1f} min)')
print(f'Frec.     : {SRATE} Hz')
print()

df.describe().round(4)

## 2. Señal completa — todos los canales

Vista rápida para detectar artefactos, saturaciones o canales ruidosos.

In [ ]:
eeg_cols = ['EEG_1', 'EEG_2']
emg_cols = ['EMG_1', 'EMG_2', 'EMG_3', 'EMG_4', 'EMG_5', 'EMG_6']
t = df['Tiempo_s'].values

fig, axes = plt.subplots(8, 1, figsize=(14, 16), sharex=True)

colors_eeg = ['#1f77b4', '#ff7f0e']
colors_emg = ['#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']

for i, col in enumerate(eeg_cols):
    axes[i].plot(t, df[col].values, color=colors_eeg[i], lw=0.4)
    axes[i].set_ylabel(col, fontsize=9)
    axes[i].set_title(f'{col}  —  EEG  |  rango: [{df[col].min():.3f}, {df[col].max():.3f}]', fontsize=8)

for j, col in enumerate(emg_cols):
    axes[j+2].plot(t, df[col].values, color=colors_emg[j], lw=0.4)
    axes[j+2].set_ylabel(col, fontsize=9)
    axes[j+2].set_title(f'{col}  —  EMG  |  rms: {np.sqrt(np.mean(df[col].values**2)):.5f}', fontsize=8)

axes[-1].set_xlabel('Tiempo (s)')
plt.suptitle('Vista completa de todos los canales', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 3. Estadísticas por canal

Comparar RMS, rango, kurtosis y skewness para identificar cuáles canales tienen mejor señal.

In [ ]:
signal_cols = eeg_cols + emg_cols
stats = []
for col in signal_cols:
    x = df[col].values
    stats.append({
        'Canal'    : col,
        'Tipo'     : 'EEG' if col.startswith('EEG') else 'EMG',
        'Media'    : np.mean(x),
        'Std'      : np.std(x),
        'RMS'      : np.sqrt(np.mean(x**2)),
        'Min'      : np.min(x),
        'Max'      : np.max(x),
        'Rango'    : np.max(x) - np.min(x),
        'Kurtosis' : kurtosis(x),
        'Skewness' : skew(x),
    })

stats_df = pd.DataFrame(stats).set_index('Canal')
stats_df.round(5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = ['#1f77b4']*2 + ['#2ca02c']*6

stats_df['RMS'].plot(kind='bar', ax=axes[0], color=colors, title='RMS por canal')
axes[0].set_ylabel('RMS (µV)')
axes[0].tick_params(axis='x', rotation=45)

stats_df['Rango'].plot(kind='bar', ax=axes[1], color=colors, title='Rango (max−min) por canal')
axes[1].set_ylabel('Amplitud (µV)')
axes[1].tick_params(axis='x', rotation=45)

stats_df['Kurtosis'].plot(kind='bar', ax=axes[2], color=colors, title='Kurtosis por canal')
axes[2].axhline(3, color='red', lw=1, ls='--', label='Gaussiana (K=3)')
axes[2].legend(fontsize=8)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Comparación estadística de canales', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Densidad espectral de potencia (PSD)

Ver en qué bandas concentra energía cada canal. Para EEG nos interesan las **bandas alfa (8–13 Hz) y beta (13–30 Hz)**. Alta kurtosis en EMG indica bursts musculares.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# EEG PSD
for i, col in enumerate(eeg_cols):
    x = df[col].values
    freqs, psd = signal.welch(x, SRATE, nperseg=2048)
    axes[0].semilogy(freqs, psd, label=col, lw=1.2, color=colors_eeg[i])

# Bandas EEG
bandas = [('Delta', 0.5, 4, '#cce5ff'), ('Theta', 4, 8, '#d4edda'),
          ('Alpha', 8, 13, '#fff3cd'), ('Beta', 13, 30, '#f8d7da'), ('Gamma', 30, 100, '#e2d9f3')]
for nombre, lo, hi, color in bandas:
    axes[0].axvspan(lo, hi, alpha=0.15, color=color, label=nombre)

axes[0].set_xlim(0, 120)
axes[0].set_title('PSD — Canales EEG')
axes[0].set_ylabel('PSD (µV²/Hz)')
axes[0].legend(fontsize=8, ncol=4)
axes[0].grid(True, alpha=0.3)

# EMG PSD
for j, col in enumerate(emg_cols):
    x = df[col].values
    freqs, psd = signal.welch(x, SRATE, nperseg=2048)
    axes[1].semilogy(freqs, psd, label=col, lw=1, color=colors_emg[j])

axes[1].set_xlim(0, 300)
axes[1].set_title('PSD — Canales EMG')
axes[1].set_xlabel('Frecuencia (Hz)')
axes[1].set_ylabel('PSD (µV²/Hz)')
axes[1].legend(fontsize=8, ncol=3)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Zoom: primeros 10 segundos

Ver la morfología de la señal a nivel de muestra para detectar ruido de línea (50 Hz), artefactos de movimiento o saturaciones.

In [ ]:
ZOOM_S = 10  # segundos a visualizar
n_zoom = int(ZOOM_S * SRATE)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for i, col in enumerate(eeg_cols):
    axes[0].plot(t[:n_zoom], df[col].values[:n_zoom], label=col, lw=0.8, color=colors_eeg[i])
axes[0].set_title(f'EEG — primeros {ZOOM_S} s')
axes[0].set_ylabel('Amplitud (µV)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for j, col in enumerate(emg_cols):
    axes[1].plot(t[:n_zoom], df[col].values[:n_zoom], label=col, lw=0.6, color=colors_emg[j], alpha=0.8)
axes[1].set_title(f'EMG — primeros {ZOOM_S} s')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud (µV)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. ¿Cuál canal EMG usar como detector de bursts?

El `app_edu.py` pide elegir **un** canal EMG. Evaluamos cuál tiene mayor varianza y kurtosis (indicadores de bursts musculares claros).

Un buen canal para burst detection tiene:
- **Alta kurtosis** → distribución con colas pesadas (hay eventos de alta amplitud bien definidos)
- **RMS alto relativo a la línea base** → buena relación señal/ruido
- **PSD con energía en 20–150 Hz** → banda típica del EMG muscular

In [ ]:
emg_ranking = []
for col in emg_cols:
    x = df[col].values
    x_centered = x - np.mean(x)
    x_rect = np.abs(x_centered)
    
    # Energía en banda EMG (20–150 Hz)
    freqs, psd = signal.welch(x_centered, SRATE, nperseg=1024)
    mask_emg = (freqs >= 20) & (freqs <= 150)
    emg_band_power = np.trapz(psd[mask_emg], freqs[mask_emg])
    total_power = np.trapz(psd, freqs)
    emg_ratio = emg_band_power / total_power
    
    emg_ranking.append({
        'Canal'           : col,
        'RMS'             : np.sqrt(np.mean(x_centered**2)),
        'Kurtosis'        : kurtosis(x_rect),
        'Potencia_EMG_band': emg_band_power,
        'Ratio_EMG_band'  : emg_ratio,
        'Score'           : kurtosis(x_rect) * emg_ratio * 100,  # score compuesto
    })

ranking_df = pd.DataFrame(emg_ranking).set_index('Canal').sort_values('Score', ascending=False)
print('Ranking de canales EMG para detección de bursts:')
print('(mayor Score = mejor candidato)\n')
ranking_df.round(4)

In [ ]:
best_emg = ranking_df.index[0]
print(f'Canal EMG recomendado para burst detection: {best_emg}')

# Visualizar el mejor canal rectificado
x_best = df[best_emg].values
x_best = x_best - np.mean(x_best)
x_rect = np.abs(x_best)
# Normalizar entre 0 y 1 (como hace app_edu.py)
x_norm = (x_rect - x_rect.min()) / (x_rect.max() - x_rect.min())

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

axes[0].plot(t, x_best, lw=0.4, color='steelblue', label=f'{best_emg} (centrado)')
axes[0].set_title(f'Canal EMG más activo: {best_emg}')
axes[0].set_ylabel('Amplitud (µV)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, x_norm, lw=0.4, color='darkorange', label='Rectificado y normalizado [0,1]')
axes[1].axhline(0.2, color='red', lw=1.2, ls='--', label='Umbral ejemplo (0.2)')
axes[1].set_title('Señal lista para umbralización (burst detection)')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud norm.')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. EEG: verificar bandas de interés

Filtrar y visualizar la **banda Beta (13–30 Hz)** en EEG_1 y EEG_2. Esta es la banda clave para detectar **desincronización pre-movimiento** (ERDS).

In [ ]:
from scipy.signal import butter, sosfilt, sosfiltfilt

def bandpass(x, lo, hi, srate, order=4):
    nyq = srate / 2
    sos = butter(order, [lo/nyq, hi/nyq], btype='band', output='sos')
    return sosfiltfilt(sos, x)

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)

bands = [
    ('Alpha (8–13 Hz)', 8, 13),
    ('Beta (13–30 Hz)', 13, 30),
]

for row, (band_name, lo, hi) in enumerate(bands):
    for col_i, eeg_col in enumerate(eeg_cols):
        x = df[eeg_col].values
        x_filt = bandpass(x, lo, hi, SRATE)
        axes[row][col_i].plot(t, x_filt, lw=0.5,
                              color=colors_eeg[col_i], alpha=0.85)
        axes[row][col_i].set_title(f'{eeg_col} — {band_name}', fontsize=9)
        axes[row][col_i].set_ylabel('Amplitud (µV)', fontsize=8)
        axes[row][col_i].grid(True, alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel('Tiempo (s)')

plt.suptitle('EEG filtrado por bandas de interés clínico', fontsize=11)
plt.tight_layout()
plt.show()

## 8. Espectrograma EEG — vista tiempo-frecuencia

Ver si hay modulación espectral a lo largo del tiempo (indicativo de actividad cognitiva/motora).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(eeg_cols):
    x = df[col].values
    f, t_spec, Sxx = signal.spectrogram(x, SRATE, nperseg=512, noverlap=256)
    # Solo frecuencias hasta 80 Hz
    freq_mask = f <= 80
    im = axes[i].pcolormesh(t_spec, f[freq_mask], 10*np.log10(Sxx[freq_mask] + 1e-12),
                             shading='gouraud', cmap='viridis')
    axes[i].set_title(f'Espectrograma {col}')
    axes[i].set_xlabel('Tiempo (s)')
    axes[i].set_ylabel('Frecuencia (Hz)')
    # Líneas de bandas
    for freq_line, label in [(8,'α'), (13,'β'), (30,'γ')]:
        axes[i].axhline(freq_line, color='white', lw=0.8, ls='--', alpha=0.6)
        axes[i].text(t_spec[-1]*1.01, freq_line, label, color='white', fontsize=8, va='center')
    plt.colorbar(im, ax=axes[i], label='dB')

plt.suptitle('Espectrograma EEG (0–80 Hz)', fontsize=11)
plt.tight_layout()
plt.show()

## 9. Resumen y recomendaciones

Basado en el análisis anterior:

In [ ]:
print('=' * 55)
print('RESUMEN DE EXPLORACIÓN')
print('=' * 55)
print(f'Duración total   : {df.Tiempo_s.iloc[-1]:.0f} s  ({df.Tiempo_s.iloc[-1]/60:.1f} min)')
print(f'Tasa de muestreo : {SRATE} Hz')
print(f'Muestras         : {len(df):,}')
print()
print('Canales EEG:')
for col in eeg_cols:
    print(f'  {col}: rango [{df[col].min():.3f}, {df[col].max():.3f}] µV')
print()
print('Canal EMG recomendado para burst detection:')
print(f'  → {best_emg}  (mayor kurtosis y energía en banda 20–150 Hz)')
print()
print('Próximos pasos:')
print('  1. Aplicar filtros (highpass 1 Hz, lowpass 100 Hz, notch 50 Hz)')
print('  2. Detectar bursts en', best_emg, 'con app_edu.py')
print('  3. Usar bursts como marcadores → segmentar EEG')
print('  4. Calcular ERP y ERDS (desincronización beta pre-movimiento)')
print('=' * 55)